## 3. Preprocess and standardize the edge table

This block transforms the raw PrimeKG edge table into a cleaner and more consistent format that can be used in the later graph-building and modeling steps.

The main purpose of this block is to produce a **canonical edge table** where every kept relation has:
- a consistent relation label
- a clearly defined source and destination node
- normalized node types
- globally unique node identifiers
- duplicate edges removed

### Step A: Select the required columns and clean the data

## Relationship Types

### Positive Relationships (`+ve`)

| # | Relation | Display Relation |
|---|----------|------------------|
| 1 | `indication` | drug → indication → disease |
| 2 | `disease_protein` | disease → associated with → protein |
| 3 | `drug_protein` | drug → carrier → protein |
| 4 | `drug_protein` | drug → enzyme → protein |
| 5 | `drug_protein` | drug → target → protein |
| 6 | `drug_protein` | drug → transporter → protein |
| 7 | `phenotype_protein` | phenotype → associated with → protein |
| 8 | `disease_phenotype_positive` | disease → phenotype present → phenotype |

### Similarity Relationships

| # | Relation | Display Relation |
|---|----------|------------------|
| 1 | `disease_disease` | disease → parent-child → disease |
| 2 | `phenotype_phenotype` | phenotype → parent-child → phenotype |
| 3 | `protein_protein` | protein → ppi → protein |

### Negative Relationships (`-ve`)

| # | Relation | Display Relation |
|---|----------|------------------|
| 1 | `contraindication` | drug → contraindication → disease |
| 2 | `drug_effect` | drug → side effect → effect |


### Step B: Filter to the relation types used in this project

PrimeKG includes many different biomedical relations, but not all of them are needed for the drug repurposing pipeline. This section keeps only the subset of relations relevant to the project, including:

- **drug–disease treatment links** such as `indication`
- **drug–disease risk links** such as `contraindication`
- **drug–protein interaction types** such as carrier, enzyme, target, and transporter
- **disease–protein** and **phenotype–protein** associations
- **disease–phenotype** links
- **disease** and **phenotype ontology hierarchy** links
- **protein–protein interaction** edges
- **drug side effect** edges

The filtering is done using valid `(relation, display_relation)` pairs so that only the intended meanings are retained. This is important because some raw relation names are broad and need the display label to clarify the exact relationship.

### Step C: Normalize node types

One issue in the raw PrimeKG data is that some node types are not specific enough. For example, the raw type `effect/phenotype` can refer either to a phenotype or to a drug side effect depending on the relation context.

To fix this, the block creates normalized node types such as:
- `drug`
- `disease`
- `protein`
- `phenotype`
- `effect`

Straightforward raw types like `drug`, `disease`, and `gene/protein` are mapped directly.  
The ambiguous `effect/phenotype` type is resolved using the relation it appears in:
- if it appears in disease or phenotype relations, it becomes `phenotype`
- if it appears in drug side-effect relations, it becomes `effect`

### Step D: Create canonical entity UIDs

After normalizing node types, the block creates a **canonical unique identifier** for every node using the format:

`<normalized_type>:<raw_id>`

For example, a drug and a disease could both have the same numeric ID in the original dataset, but once the type is added, they become distinct identifiers.

### Step E: Standardize edge labels and enforce direction

This is one of the most important parts of the block.

First, each `(relation, display_relation)` pair is mapped to a single **canonical edge label**. For example:
- `indication` stays as `indication`
- `drug_protein` with `target` becomes `drug_target_protein`
- `disease_protein` with `associated with` becomes `disease_associated_with_protein`

This creates a cleaner and more explicit edge vocabulary.

Next, the block defines a **canonical direction** for every edge label. For example:
- `indication` should always go from **drug → disease**
- `disease_associated_with_protein` should always go from **disease → protein**
- `drug_has_side_effect` should always go from **drug → effect**

However, in the raw data, some rows may appear in the opposite direction. To fix this, the code checks whether each row is already in the correct order or needs to be flipped. If necessary, the source and destination columns are swapped so that every retained edge follows the same direction convention.

A boolean column called `is_flipped` is stored so it is still possible to tell which rows had to be reversed during preprocessing.

This step is essential because later graph construction and model training depend on consistent edge direction.

### Step F: Deduplicate edges

After canonicalization, some edges can still appear more than once. This often happens because:
- the raw file may include repeated rows
- an edge may have existed in both directions before standardization
- some relations, such as protein–protein interaction, are naturally undirected

To handle this, the block creates a deduplication key based on:
- the canonical edge label
- the source UID
- the destination UID

For undirected relations like protein–protein interactions, the source and destination UIDs are first sorted so that `(A, B)` and `(B, A)` are treated as the same edge.

Once duplicates are removed, each remaining edge is assigned a stable `canonical_edge_id`.

In [ ]:
# Preprocess and standardize

assert "raw_df" in globals(), "raw_df not found — run Block 02 first."

# A. column selection and type coercion

# Keep only the columns used later
REQUIRED_COLUMNS = [
    "relation", "display_relation",
    "x_index", "x_id", "x_type", "x_name", "x_source",
    "y_index", "y_id", "y_type", "y_name", "y_source",
]

# Validate the expected schema
missing_cols = [c for c in REQUIRED_COLUMNS if c not in raw_df.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns in kg.csv: {missing_cols}")

# make a working copy
df = raw_df[REQUIRED_COLUMNS].copy()

# Clean string columns
df["x_index"] = pd.to_numeric(df["x_index"], errors="coerce")
df["y_index"] = pd.to_numeric(df["y_index"], errors="coerce")

STRING_COLS = [c for c in REQUIRED_COLUMNS if c not in ("x_index", "y_index")]
df[STRING_COLS] = df[STRING_COLS].astype(str).apply(lambda s: s.str.strip())

# Preserve the original row index for traceability
df["raw_row_id"] = df.index

print(f"[A] Column selection done — shape: {df.shape}")


# B.relation + display_relation filtering

# Keep only the necessary relation types
RELATION_DISPLAY_RULES: dict[str, set] = {
    # Therapeutic
    "indication":              {"indication"},
    "contraindication":        {"contraindication"},
    # Drug mechanism
    "drug_protein":            {"carrier", "enzyme", "target", "transporter"},
    # Disease / phenotype biology
    "disease_protein":         {"associated with"},
    "phenotype_protein":       {"associated with"},
    "disease_phenotype_positive": {"phenotype present"},
    # Ontology hierarchies + PPI
    "disease_disease":         {"parent-child"},
    "phenotype_phenotype":     {"parent-child"},
    "protein_protein":         {"ppi"},
    # Safety
    "drug_effect":             {"side effect"},
}

# Build the filter using a vectorized merge instead of row-wise apply.
# Create a set of (relation, display_relation) valid pairs for fast lookup.
valid_pairs = {
    (rel, disp)
    for rel, disps in RELATION_DISPLAY_RULES.items()
    for disp in disps
}

keep_mask = df.apply(
    lambda row: (row["relation"], row["display_relation"]) in valid_pairs,
    axis=1,
)
df = df[keep_mask].copy().reset_index(drop=True)

print(f"[B] Relation filtering done — shape: {df.shape}")
print(f"    Relation counts:")
print(df["relation"].value_counts().to_string())


# C. Node type normalization

# Direct type map for the three unambiguous raw types
_SIMPLE_TYPE_MAP = {"drug": "drug", "disease": "disease", "gene/protein": "protein"}

df["x_type_norm"] = df["x_type"].map(_SIMPLE_TYPE_MAP)
df["y_type_norm"] = df["y_type"].map(_SIMPLE_TYPE_MAP)

# Resolve "effect/phenotype" by relation context
# Build a lookup: (relation, raw_type) → normalized_type
_EFFECT_PHENO_MAP: dict[tuple, str] = {
    ("disease_phenotype_positive", "effect/phenotype"): "phenotype",
    ("phenotype_protein",          "effect/phenotype"): "phenotype",
    ("phenotype_phenotype",        "effect/phenotype"): "phenotype",
    ("drug_effect",                "effect/phenotype"): "effect",
}

for (rel, raw_type), norm_type in _EFFECT_PHENO_MAP.items():
    rel_mask = df["relation"] == rel
    df.loc[rel_mask & (df["x_type"] == raw_type), "x_type_norm"] = norm_type
    df.loc[rel_mask & (df["y_type"] == raw_type), "y_type_norm"] = norm_type

# Validate: no row should be left unmapped
unmapped_x = df["x_type_norm"].isna().sum()
unmapped_y = df["y_type_norm"].isna().sum()
if unmapped_x or unmapped_y:
    bad = df[df["x_type_norm"].isna() | df["y_type_norm"].isna()][
        ["raw_row_id", "relation", "x_type", "y_type"]
    ]
    raise ValueError(
        f"Node type normalization failed: {unmapped_x} unmapped x-side, "
        f"{unmapped_y} unmapped y-side.\nSample:\n{bad.head()}"
    )

print(f"[C] Node type normalization done.")
print(f"    x_type_norm unique: {sorted(df['x_type_norm'].unique())}")
print(f"    y_type_norm unique: {sorted(df['y_type_norm'].unique())}")


# D. Canonical entity uids

df["x_uid"] = df["x_type_norm"] + ":" + df["x_id"].astype(str)
df["y_uid"] = df["y_type_norm"] + ":" + df["y_id"].astype(str)

# No UID should be empty or contain NaN
assert not df["x_uid"].str.contains("nan", na=True).any(), "x_uid contains NaN"
assert not df["y_uid"].str.contains("nan", na=True).any(), "y_uid contains NaN"

# Each UID maps to exactly one normalized type
for side, uid_col, type_col in [("x", "x_uid", "x_type_norm"), ("y", "y_uid", "y_type_norm")]:
    multi_type = df.groupby(uid_col)[type_col].nunique()
    bad = multi_type[multi_type > 1]
    if not bad.empty:
        raise ValueError(f"{side}_uid maps to multiple types: {bad.head()}")

print(f"[D] Canonical UIDs created.")
print(f"    Unique x_uid: {df['x_uid'].nunique():,}")
print(f"    Unique y_uid: {df['y_uid'].nunique():,}")
print(f"    Total unique nodes: {pd.concat([df['x_uid'], df['y_uid']]).nunique():,}")


# E. Canonical edge labels + direction enforcement

# Map (relation, display_relation) → canonical edge label
_EDGE_LABEL_MAP: dict[tuple, str] = {
    ("indication",                  "indication"):      "indication",
    ("contraindication",            "contraindication"): "contraindication",
    ("disease_protein",             "associated with"): "disease_associated_with_protein",
    ("phenotype_protein",           "associated with"): "phenotype_associated_with_protein",
    ("disease_phenotype_positive",  "phenotype present"): "disease_has_phenotype",
    ("disease_disease",             "parent-child"):    "disease_parent_child_disease",
    ("phenotype_phenotype",         "parent-child"):    "phenotype_parent_child_phenotype",
    ("protein_protein",             "ppi"):             "protein_interacts_with_protein",
    ("drug_effect",                 "side effect"):     "drug_has_side_effect",
    ("drug_protein",                "carrier"):         "drug_carrier_protein",
    ("drug_protein",                "enzyme"):          "drug_enzyme_protein",
    ("drug_protein",                "target"):          "drug_target_protein",
    ("drug_protein",                "transporter"):     "drug_transporter_protein",
}

df["edge_label"] = df.apply(
    lambda row: _EDGE_LABEL_MAP.get((row["relation"], row["display_relation"])),
    axis=1,
)

# Validate all rows got a label
if df["edge_label"].isna().any():
    bad = df[df["edge_label"].isna()][["raw_row_id", "relation", "display_relation"]].head()
    raise ValueError(
        f"Some rows got no edge label:\n{bad}"
    )

# Define the one canonical (src_type, dst_type) per edge label
_CANONICAL_DIRECTION: dict[str, tuple] = {
    "indication":                         ("drug",      "disease"),
    "contraindication":                   ("drug",      "disease"),
    "disease_associated_with_protein":    ("disease",   "protein"),
    "drug_carrier_protein":               ("drug",      "protein"),
    "drug_enzyme_protein":                ("drug",      "protein"),
    "drug_target_protein":                ("drug",      "protein"),
    "drug_transporter_protein":           ("drug",      "protein"),
    "phenotype_associated_with_protein":  ("phenotype", "protein"),
    "disease_has_phenotype":              ("disease",   "phenotype"),
    "disease_parent_child_disease":       ("disease",   "disease"),
    "phenotype_parent_child_phenotype":   ("phenotype", "phenotype"),
    "protein_interacts_with_protein":     ("protein",   "protein"),
    "drug_has_side_effect":               ("drug",      "effect"),
}

# Vectorized direction enforcement

# Add a helper column: expected src_type for each row's edge_label
df["_exp_src"] = df["edge_label"].map(lambda lbl: _CANONICAL_DIRECTION[lbl][0])
df["_exp_dst"] = df["edge_label"].map(lambda lbl: _CANONICAL_DIRECTION[lbl][1])

# Rows that are already in canonical order
already_ok  = (df["x_type_norm"] == df["_exp_src"]) & (df["y_type_norm"] == df["_exp_dst"])
# Rows that need to be flipped
needs_flip  = (df["x_type_norm"] == df["_exp_dst"]) & (df["y_type_norm"] == df["_exp_src"])

# Rows that match neither direction
neither = ~(already_ok | needs_flip)
if neither.any():
    bad_rows = df[neither][["raw_row_id", "edge_label", "x_type_norm", "y_type_norm"]].head()
    raise ValueError(
        f"Found rows where neither x→y nor y→x matches canonical direction:\n{bad_rows}"
    )

# Build canonical src/dst columns using np.where for the swap — O(rows), no loops
def _swap_col(col_ok, col_flipped):
    """Return col_ok where already_ok, col_flipped where needs_flip."""
    return np.where(already_ok, col_ok, col_flipped)

df["src_uid"]       = _swap_col(df["x_uid"],       df["y_uid"])
df["src_type"]      = _swap_col(df["x_type_norm"],  df["y_type_norm"])
df["src_id_raw"]    = _swap_col(df["x_id"],         df["y_id"])
df["src_name"]      = _swap_col(df["x_name"],       df["y_name"])
df["src_source"]    = _swap_col(df["x_source"],     df["y_source"])
df["src_index_raw"] = _swap_col(df["x_index"],      df["y_index"])

df["dst_uid"]       = _swap_col(df["y_uid"],        df["x_uid"])
df["dst_type"]      = _swap_col(df["y_type_norm"],  df["x_type_norm"])
df["dst_id_raw"]    = _swap_col(df["y_id"],         df["x_id"])
df["dst_name"]      = _swap_col(df["y_name"],       df["x_name"])
df["dst_source"]    = _swap_col(df["y_source"],     df["x_source"])
df["dst_index_raw"] = _swap_col(df["y_index"],      df["x_index"])

df["is_flipped"] = needs_flip

# Drop the helper columns
df.drop(columns=["_exp_src", "_exp_dst"], inplace=True)

print(f"[E] Edge labels + direction done.")
print(f"    Rows kept as-is : {already_ok.sum():,}")
print(f"    Rows flipped     : {needs_flip.sum():,}")


# F. Edge deduplication

# Undirected edge types: deduplicate (A, B) and (B, A) as the same edge
UNDIRECTED_LABELS = {"protein_interacts_with_protein"}

# For undirected edges, sort UID pair lexicographically before hashing
dedup_src = df["src_uid"].copy()
dedup_dst = df["dst_uid"].copy()

undirected_mask = df["edge_label"].isin(UNDIRECTED_LABELS)
dedup_src[undirected_mask] = df.loc[undirected_mask, ["src_uid", "dst_uid"]].min(axis=1)
dedup_dst[undirected_mask] = df.loc[undirected_mask, ["src_uid", "dst_uid"]].max(axis=1)

# Build a single string dedup key per row
df["_dedup_key"] = df["edge_label"] + "||" + dedup_src + "||" + dedup_dst

n_before = len(df)
df = df.drop_duplicates(subset="_dedup_key", keep="first").copy()
df.drop(columns=["_dedup_key"], inplace=True)
df.reset_index(drop=True, inplace=True)

# Give each unique canonical edge a stable ID
df["canonical_edge_id"] = df.index

n_removed = n_before - len(df)
print(f"[F] Deduplication done.")
print(f"    Edges before : {n_before:,}")
print(f"    Edges removed: {n_removed:,}")
print(f"    Edges after  : {len(df):,}")


# Final output

canonical_edges_df = df[
    [
        "canonical_edge_id",
        "raw_row_id",
        "edge_label",
        "src_uid",  "src_type",  "src_id_raw",  "src_name",  "src_source",  "src_index_raw",
        "dst_uid",  "dst_type",  "dst_id_raw",  "dst_name",  "dst_source",  "dst_index_raw",
        "is_flipped",
    ]
].copy()

print("\n" + "=" * 55)
print("  CANONICAL EDGES TABLE COMPLETE")
print("=" * 55)
print(f"  Total edges : {len(canonical_edges_df):,}")
print()
print("  Edge label counts:")
print(canonical_edges_df["edge_label"].value_counts().to_string())
print()
print("  Type signatures  (src_type → edge_label → dst_type):")
sig = (
    canonical_edges_df
    .groupby(["src_type", "edge_label", "dst_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["src_type", "edge_label"])
)
display(sig)
print()
print("  Sample rows:")
display(canonical_edges_df.head(10))

[A] Column selection done — shape: (8100498, 13)
[B] Relation filtering done — shape: (1473126, 13)
    Relation counts:
relation
protein_protein               642150
disease_phenotype_positive    300634
disease_protein               160822
drug_effect                   129568
disease_disease                64388
contraindication               61350
drug_protein                   51306
phenotype_phenotype            37472
indication                     18776
phenotype_protein               6660
[C] Node type normalization done.
    x_type_norm unique: ['disease', 'drug', 'effect', 'phenotype', 'protein']
    y_type_norm unique: ['disease', 'drug', 'effect', 'phenotype', 'protein']
[D] Canonical UIDs created.
    Unique x_uid: 59,121
    Unique y_uid: 59,121
    Total unique nodes: 59,121
[E] Edge labels + direction done.
    Rows kept as-is : 1,108,568
    Rows flipped     : 1,108,568
[F] Deduplication done.
    Edges before : 1,473,126
    Edges removed: 685,633
    Edges after  : 787

,src_type,edge_label,dst_type,count
0,disease,disease_associated_with_protein,protein,80411
1,disease,disease_has_phenotype,phenotype,150317
2,disease,disease_parent_child_disease,disease,64388
3,drug,contraindication,disease,30675
4,drug,drug_carrier_protein,protein,864
5,drug,drug_enzyme_protein,protein,5317
6,drug,drug_has_side_effect,effect,64784
7,drug,drug_target_protein,protein,16380
8,drug,drug_transporter_protein,protein,3092
9,drug,indication,disease,9388



  Sample rows:


,canonical_edge_id,raw_row_id,edge_label,src_uid,src_type,src_id_raw,src_name,src_source,src_index_raw,dst_uid,dst_type,dst_id_raw,dst_name,dst_source,dst_index_raw,is_flipped
0,0,0,protein_interacts_with_protein,protein:9796,protein,9796,PHYHIP,NCBI,0,protein:56992,protein,56992,KIF15,NCBI,8889,True
1,1,1,protein_interacts_with_protein,protein:7918,protein,7918,GPANK1,NCBI,1,protein:9240,protein,9240,PNMA1,NCBI,2798,True
2,2,2,protein_interacts_with_protein,protein:8233,protein,8233,ZRSR2,NCBI,2,protein:23548,protein,23548,TTC33,NCBI,5646,True
3,3,3,protein_interacts_with_protein,protein:4899,protein,4899,NRF1,NCBI,3,protein:11253,protein,11253,MAN1B1,NCBI,11592,True
4,4,4,protein_interacts_with_protein,protein:5297,protein,5297,PI4KA,NCBI,4,protein:8601,protein,8601,RGS20,NCBI,2122,True
5,5,5,protein_interacts_with_protein,protein:6564,protein,6564,SLC15A1,NCBI,5,protein:8933,protein,8933,RTL8C,NCBI,2352,True
6,6,6,protein_interacts_with_protein,protein:8668,protein,8668,EIF3I,NCBI,6,protein:22976,protein,22976,PAXIP1,NCBI,5164,True
7,7,7,protein_interacts_with_protein,protein:10826,protein,10826,FAXDC2,NCBI,7,protein:345274,protein,345274,SLC10A6,NCBI,3934,True
8,8,8,protein_interacts_with_protein,protein:4489,protein,4489,MT1A,NCBI,8,protein:7157,protein,7157,TP53,NCBI,1785,True
9,9,9,protein_interacts_with_protein,protein:6272,protein,6272,SORT1,NCBI,9,protein:54873,protein,54873,PALMD,NCBI,13895,True


## 4. Build the node and edge tables

## 4. Build the master node table

This block creates a unified node table from the canonical edge table produced in Block 3. Instead of repeatedly pulling node information from the source and destination columns of the edge table, this block creates a separate node-level reference table where each node appears only once.

### A. Extract nodes from the canonical edge table

The block starts by checking that `canonical_edges_df` is available. It then pulls node information from both sides of each edge:
- the source node columns
- the destination node columns

Since source and destination nodes are stored in parallel columns, both sets are renamed into the same schema and then combined into one dataframe. This gives a complete list of all nodes that appear anywhere in the processed graph.

### B. Remove duplicates and validate node consistency

After combining source and destination nodes, the block removes duplicate entries so that each `node_uid` appears only once in the final node table. The `node_uid` created earlier is used as the main identifier because it is globally unique across all node types.

The block also checks that each node UID maps cleanly to one set of metadata. This helps confirm that the preprocessing from Block 3 worked correctly and that no node is associated with conflicting names, types, or sources.


### C. Finalize the master node dataframe

Once the unique nodes have been extracted and validated, the block stores them in a final dataframe called `nodes_master_df`. This dataframe contains one row per unique node and serves as the node-side companion to `canonical_edges_df`.

The completed node table makes later steps much cleaner because all graph entities are now stored separately from the edge list.

In [ ]:
# Master Node Table

assert "canonical_edges_df" in globals(), "canonical_edges_df not found — run Block 03 first."

# A. build master node table

# Pull node records from each side of the edge table
src_records = canonical_edges_df[[
    "src_uid", "src_type", "src_id_raw", "src_name", "src_source", "src_index_raw"
]].rename(columns={
    "src_uid": "node_uid", "src_type": "node_type",
    "src_id_raw": "node_id_raw", "src_name": "node_name",
    "src_source": "node_source", "src_index_raw": "node_index_raw",
})

dst_records = canonical_edges_df[[
    "dst_uid", "dst_type", "dst_id_raw", "dst_name", "dst_source", "dst_index_raw"
]].rename(columns={
    "dst_uid": "node_uid", "dst_type": "node_type",
    "dst_id_raw": "node_id_raw", "dst_name": "node_name",
    "dst_source": "node_source", "dst_index_raw": "node_index_raw",
})

# Merge all candidate node records
all_records = pd.concat([src_records, dst_records], ignore_index=True)

# Clean strings
for col in ["node_uid", "node_type", "node_id_raw", "node_name", "node_source"]:
    all_records[col] = all_records[col].astype(str).str.strip()

# Each UID should map to exactly one node_type and one node_id_raw.
type_per_uid = all_records.groupby("node_uid")["node_type"].nunique()
id_per_uid   = all_records.groupby("node_uid")["node_id_raw"].nunique()

bad_type = type_per_uid[type_per_uid > 1]
bad_id   = id_per_uid[id_per_uid > 1]
if not bad_type.empty or not bad_id.empty:
    raise ValueError(
        f"UIDs mapping to multiple types: {len(bad_type)}\n"
        f"UIDs mapping to multiple raw IDs: {len(bad_id)}"
    )

# Resolve to one preferred record per UID.
# Preference order:
#   1. Non-grouped source first
#   2. Non-empty name first
#   3. Alphabetically by name
#   4. Alphabetically by source
all_records["_is_grouped"] = all_records["node_source"].str.contains(
    "grouped", case=False, na=False
).astype(int)                                   # 0 = not grouped
all_records["_name_empty"] = (all_records["node_name"] == "").astype(int)

all_records_sorted = all_records.sort_values(
    by=["node_uid", "_is_grouped", "_name_empty", "node_name", "node_source"],
    ascending=[True, True, True, True, True],
)
nodes_master_df = all_records_sorted.drop_duplicates(subset="node_uid", keep="first").copy()
nodes_master_df.drop(columns=["_is_grouped", "_name_empty"], inplace=True)

# Add a normalized name for fuzzy search in later blocks
nodes_master_df["node_name_normalized"] = (
    nodes_master_df["node_name"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace("_", " ")
)

# Assign per-type integer indices
nodes_master_df = (
    nodes_master_df
    .sort_values(["node_type", "node_uid"])
    .reset_index(drop=True)
)
nodes_master_df["node_index_within_type"] = (
    nodes_master_df.groupby("node_type").cumcount()
)
nodes_master_df["node_index_global"] = nodes_master_df.index

# Final column order
nodes_master_df = nodes_master_df[[
    "node_uid", "node_type", "node_id_raw", "node_name",
    "node_name_normalized", "node_source",
    "node_index_raw", "node_index_within_type", "node_index_global",
]].copy()

print("=" * 55)
print("  MASTER NODE TABLE")
print("=" * 55)
print(f"  Total unique nodes : {len(nodes_master_df):,}")
print()
print("  Nodes per type:")
vc = nodes_master_df["node_type"].value_counts()
for ntype, cnt in vc.items():
    print(f"    {ntype}: {cnt:,}")
print()
print("  Sample:")
display(nodes_master_df.head(10))


# B. Build lookup dictionaries

# These are used in all model blocks and evidence blocks.
uid_to_index_dict   = {}
index_to_uid_dict   = {}
index_to_name_dict  = {}
index_to_source_dict= {}

for node_type, group in nodes_master_df.groupby("node_type"):
    uid_to_index_dict[node_type]    = dict(zip(group["node_uid"], group["node_index_within_type"]))
    index_to_uid_dict[node_type]    = dict(zip(group["node_index_within_type"], group["node_uid"]))
    index_to_name_dict[node_type]   = dict(zip(group["node_index_within_type"], group["node_name"]))
    index_to_source_dict[node_type] = dict(zip(group["node_index_within_type"], group["node_source"]))

# num_nodes_dict is the standard PyG input
num_nodes_dict: dict[str, int] = (
    nodes_master_df.groupby("node_type")["node_index_within_type"]
    .max()
    .add(1)
    .astype(int)
    .to_dict()
)

print("\n  num_nodes_dict:")
for k, v in sorted(num_nodes_dict.items()):
    print(f"    {k}: {v:,}")


# C. Build model-ready edge table

edges = canonical_edges_df.copy()

# Map UIDs → integer indices using the lookup dicts built above
uid_to_index_flat = {}
for node_type, mapping in uid_to_index_dict.items():
    uid_to_index_flat.update(mapping)

edges["src_index"] = edges["src_uid"].map(uid_to_index_flat)
edges["dst_index"] = edges["dst_uid"].map(uid_to_index_flat)

# Every edge should have found an index
missing_src = edges["src_index"].isna().sum()
missing_dst = edges["dst_index"].isna().sum()
if missing_src or missing_dst:
    raise ValueError(
        f"Failed to map indices: {missing_src} missing src_index, "
        f"{missing_dst} missing dst_index.\n"
        "This means some nodes in canonical_edges_df are absent from nodes_master_df."
    )

edges["src_index"] = edges["src_index"].astype(int)
edges["dst_index"] = edges["dst_index"].astype(int)

# Add PyG-style edge triplet
edges["edge_triplet"] = list(zip(edges["src_type"], edges["edge_label"], edges["dst_type"]))
edges["edge_triplet_str"] = (
    "(" + edges["src_type"] + ", " + edges["edge_label"] + ", " + edges["dst_type"] + ")"
)

# Sort for stable downstream behavior
edges_model_ready_df = (
    edges[[
        "canonical_edge_id", "raw_row_id",
        "edge_label", "edge_triplet", "edge_triplet_str",
        "src_uid", "src_type", "src_index", "src_name", "src_source",
        "dst_uid", "dst_type", "dst_index", "dst_name", "dst_source",
        "is_flipped",
    ]]
    .sort_values(["src_type", "edge_label", "dst_type", "src_index", "dst_index"])
    .reset_index(drop=True)
    .copy()
)
edges_model_ready_df["model_edge_id"] = edges_model_ready_df.index

print("\n" + "=" * 55)
print("  MODEL-READY EDGE TABLE")
print("=" * 55)
print(f"  Total model-ready edges : {len(edges_model_ready_df):,}")
print()
print("  Edge counts by triplet:")
triplet_summary = (
    edges_model_ready_df
    .groupby(["src_type", "edge_label", "dst_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["src_type", "edge_label"])
)
display(triplet_summary)
print()
print("  Sample rows:")
display(edges_model_ready_df.head(10))
print()
print("=" * 55)
print("  BLOCK 04 COMPLETE — objects available:")
print("    nodes_master_df")
print("    edges_model_ready_df")
print("    num_nodes_dict")
print("    uid_to_index_dict")
print("    index_to_name_dict")
print("    index_to_source_dict")
print("=" * 55)

  MASTER NODE TABLE
  Total unique nodes : 59,121

  Nodes per type:
    protein: 19,059
    disease: 17,080
    phenotype: 15,311
    drug: 6,681
    effect: 990

  Sample:


,node_uid,node_type,node_id_raw,node_name,node_name_normalized,node_source,node_index_raw,node_index_within_type,node_index_global
0,disease:1,disease,1,disease or disorder,disease or disorder,MONDO,35856,0,0
1,disease:1000,disease,1000,mixed mineral dust pneumoconiosis,mixed mineral dust pneumoconiosis,MONDO,35973,1,1
2,disease:10000,disease,10000,"rod-cone dystrophy, sensorineural deafness, and Fanconi-type renal dysfunction","rod-cone dystrophy, sensorineural deafness, and fanconi-type renal dysfunction",MONDO,97679,2,2
3,disease:100000,disease,100000,MED12-related intellectual disability syndrome,med12-related intellectual disability syndrome,MONDO,38823,3,3
4,disease:100001,disease,100001,alpha-gal syndrome,alpha-gal syndrome,MONDO,99958,4,4
5,disease:100002,disease,100002,food protein-induced allergic proctocolitis,food protein-induced allergic proctocolitis,MONDO,99959,5,5
6,disease:100004,disease,100004,mast cell activation syndrome,mast cell activation syndrome,MONDO,39866,6,6
7,disease:100005,disease,100005,primary mast cell activation syndrome,primary mast cell activation syndrome,MONDO,99960,7,7
8,disease:100006,disease,100006,secondary mast cell activation syndrome,secondary mast cell activation syndrome,MONDO,99961,8,8
9,disease:100008,disease,100008,food protein-induced enterocolitis syndrome,food protein-induced enterocolitis syndrome,MONDO,99962,9,9



  num_nodes_dict:
    disease: 17,080
    drug: 6,681
    effect: 990
    phenotype: 15,311
    protein: 19,059

  MODEL-READY EDGE TABLE
  Total model-ready edges : 787,493

  Edge counts by triplet:


,src_type,edge_label,dst_type,count
0,disease,disease_associated_with_protein,protein,80411
1,disease,disease_has_phenotype,phenotype,150317
2,disease,disease_parent_child_disease,disease,64388
3,drug,contraindication,disease,30675
4,drug,drug_carrier_protein,protein,864
5,drug,drug_enzyme_protein,protein,5317
6,drug,drug_has_side_effect,effect,64784
7,drug,drug_target_protein,protein,16380
8,drug,drug_transporter_protein,protein,3092
9,drug,indication,disease,9388



  Sample rows:


,canonical_edge_id,raw_row_id,edge_label,edge_triplet,edge_triplet_str,src_uid,src_type,src_index,src_name,src_source,dst_uid,dst_type,dst_index,dst_name,dst_source,is_flipped,model_edge_id
0,616642,3293050,disease_associated_with_protein,"(disease, disease_associated_with_protein, protein)","(disease, disease_associated_with_protein, protein)",disease:10001_19287_23046_23048,disease,17,ectodermal dysplasia syndrome,MONDO_grouped,protein:10804,protein,1104,GJB6,NCBI,True,0
1,630295,3306703,disease_associated_with_protein,"(disease, disease_associated_with_protein, protein)","(disease, disease_associated_with_protein, protein)",disease:10001_19287_23046_23048,disease,17,ectodermal dysplasia syndrome,MONDO_grouped,protein:128178,protein,2270,EDARADD,NCBI,True,1
2,630287,3306695,disease_associated_with_protein,"(disease, disease_associated_with_protein, protein)","(disease, disease_associated_with_protein, protein)",disease:10001_19287_23046_23048,disease,17,ectodermal dysplasia syndrome,MONDO_grouped,protein:1896,protein,3881,EDA,NCBI,True,2
3,616710,3293118,disease_associated_with_protein,"(disease, disease_associated_with_protein, protein)","(disease, disease_associated_with_protein, protein)",disease:10001_19287_23046_23048,disease,17,ectodermal dysplasia syndrome,MONDO_grouped,protein:2706,protein,6215,GJB2,NCBI,True,3
4,616639,3293047,disease_associated_with_protein,"(disease, disease_associated_with_protein, protein)","(disease, disease_associated_with_protein, protein)",disease:10001_19287_23046_23048,disease,17,ectodermal dysplasia syndrome,MONDO_grouped,protein:3691,protein,8165,ITGB4,NCBI,True,4
5,616638,3293046,disease_associated_with_protein,"(disease, disease_associated_with_protein, protein)","(disease, disease_associated_with_protein, protein)",disease:10001_19287_23046_23048,disease,17,ectodermal dysplasia syndrome,MONDO_grouped,protein:387,protein,8441,RHOA,NCBI,True,5
6,616640,3293048,disease_associated_with_protein,"(disease, disease_associated_with_protein, protein)","(disease, disease_associated_with_protein, protein)",disease:10001_19287_23046_23048,disease,17,ectodermal dysplasia syndrome,MONDO_grouped,protein:4953,protein,9897,ODC1,NCBI,True,6
7,616767,3293175,disease_associated_with_protein,"(disease, disease_associated_with_protein, protein)","(disease, disease_associated_with_protein, protein)",disease:10001_19287_23046_23048,disease,17,ectodermal dysplasia syndrome,MONDO_grouped,protein:5339,protein,10810,PLEC,NCBI,True,7
8,616771,3293179,disease_associated_with_protein,"(disease, disease_associated_with_protein, protein)","(disease, disease_associated_with_protein, protein)",disease:10001_19287_23046_23048,disease,17,ectodermal dysplasia syndrome,MONDO_grouped,protein:54567,protein,11075,DLL4,NCBI,True,8
9,630291,3306699,disease_associated_with_protein,"(disease, disease_associated_with_protein, protein)","(disease, disease_associated_with_protein, protein)",disease:10001_19287_23046_23048,disease,17,ectodermal dysplasia syndrome,MONDO_grouped,protein:60401,protein,13244,EDA2R,NCBI,True,9



  BLOCK 04 COMPLETE — objects available:
    nodes_master_df
    edges_model_ready_df
    num_nodes_dict
    uid_to_index_dict
    index_to_name_dict
    index_to_source_dict
